# Exercise for Unit 4.1 Naïve Bayes

### Dataset

In [1]:
documents = [
    ("Free money now!!!",                      "SPAM"),
    ("Hi mom, how are you?",                   "HAM"),
    ("Lowest price for your meds",             "SPAM"),
    ("Are we still on for dinner?",            "HAM"),
    ("Win a free iPhone today",                "SPAM"),
    ("Let's catch up tomorrow at the office",  "HAM"),
    ("Meeting at 3 PM tomorrow",               "HAM"),
    ("Get 50% off, limited time!",             "SPAM"),
    ("Team meeting in the office",             "HAM"),
    ("Click here for prizes!",                 "SPAM"),
    ("Can you send the report?",               "HAM"),
]

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
]

train_docs   = [doc for doc, _ in documents]
train_labels = [label for _, label in documents]

print(f"{'Document':<45} {'Class'}")
print("-" * 55)
for doc, label in documents:
    print(f"{doc:<45} {label}")

print(f"\nTotal documents : {len(documents)}")
print(f"SPAM            : {train_labels.count('SPAM')}")
print(f"HAM             : {train_labels.count('HAM')}")

Document                                      Class
-------------------------------------------------------
Free money now!!!                             SPAM
Hi mom, how are you?                          HAM
Lowest price for your meds                    SPAM
Are we still on for dinner?                   HAM
Win a free iPhone today                       SPAM
Let's catch up tomorrow at the office         HAM
Meeting at 3 PM tomorrow                      HAM
Get 50% off, limited time!                    SPAM
Team meeting in the office                    HAM
Click here for prizes!                        SPAM
Can you send the report?                      HAM

Total documents : 11
SPAM            : 5
HAM             : 6


---
## 1.	Performing it manually. In manually developing a Naïve Bayes model, create methods that will do the following:

### a. Generate a Bag of Words (for word frequency)

The tokenizer lowercases text and removes punctuation. The bag of words counts how often each word appears per class.

In [2]:
import re
import math
from collections import defaultdict


def tokenize(text):
    """
    Lowercase the text and return a list of word tokens.
    Punctuation and numbers are removed.
    Example: 'Free money now!!!' -> ['free', 'money', 'now']
    """
    text = text.lower()
    tokens = re.findall(r"[a-z']+", text)
    tokens = [t.strip("'") for t in tokens]
    tokens = [t for t in tokens if t]
    return tokens


def build_bag_of_words(documents):
    """
    Build a bag-of-words from the labeled documents.

    Returns:
        bow_per_class    : {class: {word: count}}
        vocabulary       : set of all unique words
        class_doc_counts : {class: number of documents}
    """
    bow_per_class    = defaultdict(lambda: defaultdict(int))
    class_doc_counts = defaultdict(int)
    all_words        = set()

    for text, label in documents:
        tokens = tokenize(text)
        class_doc_counts[label] += 1
        for token in tokens:
            bow_per_class[label][token] += 1
            all_words.add(token)

    return dict(bow_per_class), all_words, dict(class_doc_counts)


bow_per_class, vocabulary, class_doc_counts = build_bag_of_words(documents)

# Show tokenizer output
print("Tokenizer examples:")
print("-" * 40)
for doc, label in documents[:3]:
    print(f"  [{label}] '{doc}'")
    print(f"         -> {tokenize(doc)}")

# Show bag of words per class
print("\nBag of Words per class:")
print("-" * 40)
for cls in ["SPAM", "HAM"]:
    total = sum(bow_per_class[cls].values())
    print(f"\n[{cls}]  (total word tokens = {total})")
    print(f"  {'Word':<18} Count")
    print("  " + "-" * 24)
    for word, count in sorted(bow_per_class[cls].items(), key=lambda x: -x[1]):
        print(f"  {word:<18} {count}")

print(f"\nVocabulary size |V| = {len(vocabulary)}")

Tokenizer examples:
----------------------------------------
  [SPAM] 'Free money now!!!'
         -> ['free', 'money', 'now']
  [HAM] 'Hi mom, how are you?'
         -> ['hi', 'mom', 'how', 'are', 'you']
  [SPAM] 'Lowest price for your meds'
         -> ['lowest', 'price', 'for', 'your', 'meds']

Bag of Words per class:
----------------------------------------

[SPAM]  (total word tokens = 21)
  Word               Count
  ------------------------
  free               2
  for                2
  money              1
  now                1
  lowest             1
  price              1
  your               1
  meds               1
  win                1
  a                  1
  iphone             1
  today              1
  get                1
  off                1
  limited            1
  time               1
  click              1
  here               1
  prizes             1

[HAM]  (total word tokens = 32)
  Word               Count
  ------------------------
  the                3
 

### b. Calculate the prior for the class HAM and SPAM

$$\hat{P}(c_j) = \frac{N_{c_j}}{N_{\text{total}}}$$

In [3]:
def calculate_priors(class_doc_counts):
    """
    Compute P(c) for each class.
    Formula: P(c_j) = N_cj / N_total
    """
    total = sum(class_doc_counts.values())
    return {label: count / total for label, count in class_doc_counts.items()}


priors = calculate_priors(class_doc_counts)
total_docs = sum(class_doc_counts.values())

print("Prior Probabilities")
print("-" * 40)
print(f"Total training documents: {total_docs}")
print()
for cls in ["SPAM", "HAM"]:
    n = class_doc_counts[cls]
    p = priors[cls]
    print(f"  P({cls}) = {n}/{total_docs} = {p:.4f}   log P({cls}) = {math.log(p):.4f}")

Prior Probabilities
----------------------------------------
Total training documents: 11

  P(SPAM) = 5/11 = 0.4545   log P(SPAM) = -0.7885
  P(HAM) = 6/11 = 0.5455   log P(HAM) = -0.6061


### c.	Calculate the likelihood of the tokens in the vocabulary with respect to the class.

$$\hat{P}(w_i \mid c) = \frac{\text{count}(w_i, c) + 1}{\sum_{w \in V} \text{count}(w, c) + |V|}$$

In [4]:
def calculate_likelihoods(bow_per_class, vocabulary):
    """
    Compute P(w | c) for every word in the vocabulary using Laplace smoothing.
    Formula: P(w_i | c) = (count(w_i, c) + 1) / (total_words_in_c + |V|)
    """
    V = len(vocabulary)
    likelihoods = {}

    for cls, word_counts in bow_per_class.items():
        total = sum(word_counts.values())
        denominator = total + V
        likelihoods[cls] = {}
        for word in vocabulary:
            likelihoods[cls][word] = (word_counts.get(word, 0) + 1) / denominator

    return likelihoods


likelihoods = calculate_likelihoods(bow_per_class, vocabulary)

V = len(vocabulary)
spam_total = sum(bow_per_class["SPAM"].values())
ham_total  = sum(bow_per_class["HAM"].values())

print("Likelihood Table  P(word | class)  with Laplace add-1 smoothing")
print(f"SPAM denominator: {spam_total} + {V} = {spam_total + V}")
print(f"HAM  denominator: {ham_total}  + {V} = {ham_total + V}")
print()
print(f"  {'Word':<18} {'count(SPAM)':>12}  {'count(HAM)':>11}  {'P(w|SPAM)':>12}  {'P(w|HAM)':>12}")
print("  " + "-" * 72)
for word in sorted(vocabulary):
    cs = bow_per_class["SPAM"].get(word, 0)
    ch = bow_per_class["HAM"].get(word, 0)
    ps = likelihoods["SPAM"][word]
    ph = likelihoods["HAM"][word]
    print(f"  {word:<18} {cs:>12}  {ch:>11}  {ps:>12.6f}  {ph:>12.6f}")

Likelihood Table  P(word | class)  with Laplace add-1 smoothing
SPAM denominator: 21 + 42 = 63
HAM  denominator: 32  + 42 = 74

  Word                count(SPAM)   count(HAM)     P(w|SPAM)      P(w|HAM)
  ------------------------------------------------------------------------
  a                             1            0      0.031746      0.013514
  are                           0            2      0.015873      0.040541
  at                            0            2      0.015873      0.040541
  can                           0            1      0.015873      0.027027
  catch                         0            1      0.015873      0.027027
  click                         1            0      0.031746      0.013514
  dinner                        0            1      0.015873      0.027027
  for                           2            1      0.047619      0.027027
  free                          2            0      0.047619      0.013514
  get                           1            0 

### d.	Determine the class of the test sentence

Words not seen during training are ignored.

In [5]:
def classify_manual(text, priors, likelihoods, vocabulary):
    """
    Classify a text using Multinomial Naive Bayes in log-space.

    Steps:
      1. Tokenize the input
      2. Remove words not in the training vocabulary
      3. For each class: score = log P(c) + sum of log P(word | c)
      4. Return the class with the highest score
    """
    tokens         = tokenize(text)
    known_tokens   = [t for t in tokens if t in vocabulary]
    unknown_tokens = [t for t in tokens if t not in vocabulary]

    scores = {}
    for cls in priors:
        score = math.log(priors[cls])
        for word in known_tokens:
            score += math.log(likelihoods[cls][word])
        scores[cls] = score

    predicted = max(scores, key=scores.get)

    print("=" * 60)
    print(f"Input: \"{text}\"")
    print("=" * 60)
    print(f"Tokens           : {tokens}")
    if unknown_tokens:
        print(f"Unknown (ignored) : {unknown_tokens}")
    print(f"Tokens scored    : {known_tokens}")
    print()

    for cls in ["SPAM", "HAM"]:
        log_prior = math.log(priors[cls])
        print(f"  Class: {cls}")
        print(f"    log P({cls}) = {log_prior:.4f}")
        log_sum = 0
        for word in known_tokens:
            lp = math.log(likelihoods[cls][word])
            log_sum += lp
            print(f"    + log P('{word}' | {cls}) = {lp:.4f}")
        print(f"    Total score = {log_prior:.4f} + {log_sum:.4f} = {scores[cls]:.4f}")
        print()

    print("Score comparison:")
    for cls in ["SPAM", "HAM"]:
        marker = "  <- predicted" if cls == predicted else ""
        print(f"  {cls}: {scores[cls]:.4f}{marker}")
    print(f"\nPredicted class: {predicted}")
    print("=" * 60)

    return predicted, scores

In [6]:
# Test sentence i: "Limited offer, click here!"
manual_result_1, manual_scores_1 = classify_manual(test_sentences[0], priors, likelihoods, vocabulary)

Input: "Limited offer, click here!"
Tokens           : ['limited', 'offer', 'click', 'here']
Unknown (ignored) : ['offer']
Tokens scored    : ['limited', 'click', 'here']

  Class: SPAM
    log P(SPAM) = -0.7885
    + log P('limited' | SPAM) = -3.4500
    + log P('click' | SPAM) = -3.4500
    + log P('here' | SPAM) = -3.4500
    Total score = -0.7885 + -10.3500 = -11.1384

  Class: HAM
    log P(HAM) = -0.6061
    + log P('limited' | HAM) = -4.3041
    + log P('click' | HAM) = -4.3041
    + log P('here' | HAM) = -4.3041
    Total score = -0.6061 + -12.9122 = -13.5183

Score comparison:
  SPAM: -11.1384  <- predicted
  HAM: -13.5183

Predicted class: SPAM


In [7]:
# Test sentence ii: "Meeting at 2 PM with the manager."
manual_result_2, manual_scores_2 = classify_manual(test_sentences[1], priors, likelihoods, vocabulary)

Input: "Meeting at 2 PM with the manager."
Tokens           : ['meeting', 'at', 'pm', 'with', 'the', 'manager']
Unknown (ignored) : ['with', 'manager']
Tokens scored    : ['meeting', 'at', 'pm', 'the']

  Class: SPAM
    log P(SPAM) = -0.7885
    + log P('meeting' | SPAM) = -4.1431
    + log P('at' | SPAM) = -4.1431
    + log P('pm' | SPAM) = -4.1431
    + log P('the' | SPAM) = -4.1431
    Total score = -0.7885 + -16.5725 = -17.3610

  Class: HAM
    log P(HAM) = -0.6061
    + log P('meeting' | HAM) = -3.2055
    + log P('at' | HAM) = -3.2055
    + log P('pm' | HAM) = -3.6109
    + log P('the' | HAM) = -2.9178
    Total score = -0.6061 + -12.9396 = -13.5457

Score comparison:
  SPAM: -17.3610
  HAM: -13.5457  <- predicted

Predicted class: HAM


---
## 2. Use the scikit-learn package to train and test a Multinomial Naïve Bayes classifer.

`CountVectorizer` handles the bag-of-words vectorization and `MultinomialNB` trains the classifier. Setting `alpha=1.0` applies the same Laplace add-1 smoothing used in Part 1.

### Vectorize with CountVectorizer

In [8]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Build the vocabulary and transform training documents into count vectors
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_docs)

vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary ({len(vocab)} words):")
print(list(vocab))

# Show the document-term matrix
print("\nDocument-Term Matrix (word counts per document):")
dtm = pd.DataFrame(X_train.toarray(), columns=vocab)
dtm.index = [f"[{l}] {d[:30]}" for d, l in zip(train_docs, train_labels)]
print(dtm.to_string())

Vocabulary (42 words):
['50', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

Document-Term Matrix (word counts per document):
                                      50  are  at  can  catch  click  dinner  for  free  get  here  hi  how  in  iphone  let  limited  lowest  meds  meeting  mom  money  now  off  office  on  pm  price  prizes  report  send  still  team  the  time  today  tomorrow  up  we  win  you  your
[SPAM] Free money now!!!               0    0   0    0      0      0       0    0     1    0     0   0    0   0       0    0        0       0     0        0    0      1    1    0       0   0   0      0       0       0     0      0     0    0     0      0         0   0   0    0    0     0
[HAM] Hi mom

### Train the Model

In [9]:
# alpha=1.0 applies Laplace add-1 smoothing
model = MultinomialNB(alpha=1.0)
model.fit(X_train, train_labels)

print("Model trained successfully.")
print(f"Classes                   : {list(model.classes_)}")
print(f"Smoothing parameter alpha : {model.alpha}")

# Priors
print("\nPrior Probabilities:")
print("-" * 40)
for cls, log_prior in zip(model.classes_, model.class_log_prior_):
    print(f"  P({cls}) = {np.exp(log_prior):.4f}   log P({cls}) = {log_prior:.4f}")

# Likelihoods
print("\nLikelihood Table  P(word | class):")
print("-" * 50)
print(f"  {'Word':<18} {'P(w|HAM)':>12}  {'P(w|SPAM)':>12}")
print("  " + "-" * 46)
for i, word in enumerate(vocab):
    p_ham  = np.exp(model.feature_log_prob_[0][i])
    p_spam = np.exp(model.feature_log_prob_[1][i])
    print(f"  {word:<18} {p_ham:>12.6f}  {p_spam:>12.6f}")

Model trained successfully.
Classes                   : [np.str_('HAM'), np.str_('SPAM')]
Smoothing parameter alpha : 1.0

Prior Probabilities:
----------------------------------------
  P(HAM) = 0.5455   log P(HAM) = -0.6061
  P(SPAM) = 0.4545   log P(SPAM) = -0.7885

Likelihood Table  P(word | class):
--------------------------------------------------
  Word                   P(w|HAM)     P(w|SPAM)
  ----------------------------------------------
  50                     0.013514      0.031746
  are                    0.040541      0.015873
  at                     0.040541      0.015873
  can                    0.027027      0.015873
  catch                  0.027027      0.015873
  click                  0.013514      0.031746
  dinner                 0.027027      0.015873
  for                    0.027027      0.047619
  free                   0.013514      0.047619
  get                    0.013514      0.031746
  here                   0.013514      0.031746
  hi               

### a.	Determine the class of the test sentence:



In [10]:
# Transform test sentences using the same vectorizer (do NOT refit)
X_test = vectorizer.transform(test_sentences)

predictions   = model.predict(X_test)
probabilities = model.predict_proba(X_test)

print("Classification Results")
print("=" * 60)
for sentence, pred, probs in zip(test_sentences, predictions, probabilities):
    print(f"Sentence : \"{sentence}\"")
    for cls, prob in zip(model.classes_, probs):
        print(f"  P({cls}) = {prob:.4f}")
    print(f"  Predicted class: {pred}")
    print()

Classification Results
Sentence : "Limited offer, click here!"
  P(HAM) = 0.0847
  P(SPAM) = 0.9153
  Predicted class: SPAM

Sentence : "Meeting at 2 PM with the manager."
  P(HAM) = 0.9784
  P(SPAM) = 0.0216
  Predicted class: HAM



---
## Comparing the results of Manual and Scikit-learn

In [13]:
manual_results = [manual_result_1, manual_result_2]
sklearn_results = list(predictions)

# Print side-by-side comparison
print("Results Comparison")
print("=" * 65)
print(f"  {'Test Sentence':<40} {'Manual':>8}  {'sklearn':>8}")
print("  " + "-" * 60)
for sentence, manual, sk in zip(test_sentences, manual_results, sklearn_results):
    match = "Match" if manual == sk else "Differ"
    print(f"  {sentence:<40} {manual:>8}  {sk:>8}   [{match}]")

Results Comparison
  Test Sentence                              Manual   sklearn
  ------------------------------------------------------------
  Limited offer, click here!                   SPAM      SPAM   [Match]
  Meeting at 2 PM with the manager.             HAM       HAM   [Match]
